# ⚡ Household Power Consumption — LSTM Time Series
## Analyse et prédiction de consommation électrique avec un réseau LSTM

| Info | Valeur |
|---|---|
| Dataset | Individual Household Electric Power Consumption (UCI) |
| Période | Décembre 2006 – Novembre 2010 |
| Fréquence | 1 mesure par minute (~2 millions de lignes) |
| Tâche | Prédiction de `Global_active_power` (série temporelle) |
| Modèle | LSTM (Long Short-Term Memory) |

> ⚠️ **GPU recommandé** : Activer le GPU dans Colab → Exécution → Modifier le type d'exécution → T4 GPU

---

## 📦 Part 1 — Data Import and Initial Exploration

In [ ]:
!pip install -q tensorflow matplotlib seaborn scikit-learn pandas numpy

In [ ]:
# === IMPORT DES LIBRAIRIES ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {
    'main':   '#185FA5',
    'second': '#1D9E75',
    'warn':   '#E24B4A',
    'amber':  '#BA7517',
    'purple': '#534AB7'
}

np.random.seed(42)
tf.random.set_seed(42)

print(f'✅ TensorFlow : {tf.__version__}')
print(f'✅ GPU disponible : {len(tf.config.list_physical_devices("GPU")) > 0}')
if tf.config.list_physical_devices('GPU'):
    print(f'   GPU : {tf.config.list_physical_devices("GPU")[0].name}')

In [ ]:
# === CHARGEMENT DU DATASET ===
# Le fichier utilise ; comme séparateur et ? pour les valeurs manquantes

try:
    df = pd.read_csv(
        'household_power_consumption.txt',
        sep=';',
        na_values=['?'],        # '?' représente les valeurs manquantes
        low_memory=False,
        parse_dates={'datetime': ['Date', 'Time']},
        dayfirst=True,          # Format DD/MM/YYYY
        index_col='datetime'
    )
    print('✅ Dataset chargé depuis le fichier local')
except FileNotFoundError:
    import io, zipfile, requests
    url = 'https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/W6D4/household_power_consumption.zip'
    r = requests.get(url, timeout=60)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    fname = [f for f in z.namelist() if f.endswith('.txt')][0]
    df = pd.read_csv(
        z.open(fname), sep=';', na_values=['?'], low_memory=False,
        parse_dates={'datetime': ['Date', 'Time']},
        dayfirst=True, index_col='datetime'
    )
    print(f'✅ Dataset chargé depuis GitHub : {fname}')

# Afficher les premières lignes
print(f'\n=== PREMIÈRES LIGNES ===')
print(df.head())

In [ ]:
# === EXPLORATION INITIALE ===
print('=== SHAPE DU DATASET ===')
print(f'  Lignes   : {df.shape[0]:,}')
print(f'  Colonnes : {df.shape[1]}')

print('\n=== TYPES DE DONNÉES ===')
print(df.dtypes)

print('\n=== PLAGE TEMPORELLE ===')
print(f'  Début : {df.index.min()}')
print(f'  Fin   : {df.index.max()}')
print(f'  Durée : {(df.index.max() - df.index.min()).days} jours (~{(df.index.max() - df.index.min()).days//365} ans)')

print('\n=== STATISTIQUES DESCRIPTIVES ===')
df.describe().round(4)

In [ ]:
# Description des colonnes
col_desc = {
    'Global_active_power':   'Puissance active globale du ménage (kW)',
    'Global_reactive_power': 'Puissance réactive globale (kW)',
    'Voltage':               'Tension moyenne (volt)',
    'Global_intensity':      'Intensité de courant moyenne (ampère)',
    'Sub_metering_1':        'Sous-compteur 1 : cuisine (Wh)',
    'Sub_metering_2':        'Sous-compteur 2 : buanderie (Wh)',
    'Sub_metering_3':        'Sous-compteur 3 : chauffe-eau & clim (Wh)'
}
print('=== DESCRIPTION DES COLONNES ===')
for col, desc in col_desc.items():
    missing = df[col].isnull().sum()
    print(f'  {col:<30s} : {desc}')
    if missing > 0:
        print(f'    ⚠️  {missing:,} valeurs manquantes ({missing/len(df)*100:.2f}%)')

---
## 🔧 Part 2 — Handling Missing Values

In [ ]:
# === IDENTIFICATION DES VALEURS MANQUANTES ===
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(4)

missing_df = pd.DataFrame({'Count': missing, 'Percentage (%)': missing_pct})
print('=== VALEURS MANQUANTES PAR COLONNE ===')
print(missing_df)
print(f'\nTotal valeurs manquantes : {missing.sum():,}')
print(f'Lignes avec au moins 1 NaN : {df.isnull().any(axis=1).sum():,}')

# Visualisation
fig, ax = plt.subplots(figsize=(10, 4))
missing_cols = missing_df[missing_df['Count'] > 0]
ax.barh(missing_cols.index, missing_cols['Percentage (%)'],
        color=COLORS['warn'], edgecolor='white')
ax.set_xlabel('% de valeurs manquantes')
ax.set_title('Valeurs manquantes par colonne (% sur ~2M lignes)', fontweight='bold')
for i, (idx, row) in enumerate(missing_cols.iterrows()):
    ax.text(row['Percentage (%)']+0.005, i,
            f"{row['Percentage (%)']:.3f}%  ({int(row['Count']):,})",
            va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# === REMPLISSAGE DES VALEURS MANQUANTES PAR LA MOYENNE ===
for col in df.columns:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        col_mean = df[col].mean()
        df[col].fillna(col_mean, inplace=True)
        print(f'  ✅ {col:<30s} : {n_missing:,} NaN → mean = {col_mean:.4f}')

# Vérification
remaining = df.isnull().sum().sum()
print(f'\n✅ Valeurs manquantes restantes : {remaining}')
print(f'✅ Dataset complet : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

---
## 📊 Part 3 — Data Visualization

In [ ]:
# === APERÇU GLOBAL DE LA SÉRIE TEMPORELLE ===
fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

# Rééchantillonnage hebdomadaire pour lisibilité
weekly = df.resample('W').mean()

axes[0].plot(weekly.index, weekly['Global_active_power'],
             color=COLORS['main'], lw=1.2, alpha=0.9)
axes[0].fill_between(weekly.index, weekly['Global_active_power'],
                      alpha=0.15, color=COLORS['main'])
axes[0].set_title('Global Active Power — Moyenne hebdomadaire (kW)', fontweight='bold')
axes[0].set_ylabel('kW')

axes[1].plot(weekly.index, weekly['Global_intensity'],
             color=COLORS['second'], lw=1.2, alpha=0.9)
axes[1].set_title('Global Intensity — Moyenne hebdomadaire (A)', fontweight='bold')
axes[1].set_ylabel('Ampère')

axes[2].stackplot(weekly.index,
                  weekly['Sub_metering_1'],
                  weekly['Sub_metering_2'],
                  weekly['Sub_metering_3'],
                  labels=['Cuisine', 'Buanderie', 'Chauffe-eau/Clim'],
                  colors=[COLORS['main'], COLORS['second'], COLORS['amber']],
                  alpha=0.7)
axes[2].set_title('Sous-compteurs — Répartition hebdomadaire (Wh)', fontweight='bold')
axes[2].set_ylabel('Wh'); axes[2].legend(loc='upper left')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[2].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=30)

plt.suptitle('Vue d\'ensemble — Consommation électrique 2006-2010',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === RÉÉCHANTILLONNAGE JOURNALIER : SOMME ET MOYENNE DE Global_active_power ===
daily_sum  = df['Global_active_power'].resample('D').sum()
daily_mean = df['Global_active_power'].resample('D').mean()

fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

# Somme journalière
axes[0].plot(daily_sum.index, daily_sum.values,
             color=COLORS['main'], lw=0.8, alpha=0.7, label='Somme journalière')
# Moyenne mobile 30 jours
rolling_sum = daily_sum.rolling(30).mean()
axes[0].plot(rolling_sum.index, rolling_sum.values,
             color=COLORS['warn'], lw=2.5, label='Moyenne mobile 30j')
axes[0].fill_between(daily_sum.index, daily_sum.values, alpha=0.1, color=COLORS['main'])
axes[0].set_title('Global Active Power — Somme journalière (kWh)', fontweight='bold')
axes[0].set_ylabel('kWh (somme)')
axes[0].legend()

# Moyenne journalière
axes[1].plot(daily_mean.index, daily_mean.values,
             color=COLORS['second'], lw=0.8, alpha=0.7, label='Moyenne journalière')
rolling_mean = daily_mean.rolling(30).mean()
axes[1].plot(rolling_mean.index, rolling_mean.values,
             color=COLORS['purple'], lw=2.5, label='Moyenne mobile 30j')
axes[1].fill_between(daily_mean.index, daily_mean.values, alpha=0.1, color=COLORS['second'])
axes[1].set_title('Global Active Power — Moyenne journalière (kW)', fontweight='bold')
axes[1].set_ylabel('kW (moyenne)')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)

plt.suptitle('Global Active Power — Somme & Moyenne journalières (rééchantillonné par jour)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Somme totale sur la période   : {daily_sum.sum():,.0f} kWh')
print(f'Moyenne journalière globale   : {daily_mean.mean():.4f} kW')
print(f'Pic journalier (somme)        : {daily_sum.max():.2f} kWh le {daily_sum.idxmax().date()}')

In [ ]:
# === MOYENNE ET ÉCART-TYPE DE Global_intensity PAR JOUR ===
daily_int_mean = df['Global_intensity'].resample('D').mean()
daily_int_std  = df['Global_intensity'].resample('D').std()

fig, ax = plt.subplots(figsize=(18, 5))

ax.plot(daily_int_mean.index, daily_int_mean.values,
        color=COLORS['main'], lw=1.2, label='Moyenne journalière')
ax.fill_between(
    daily_int_mean.index,
    daily_int_mean - daily_int_std,
    daily_int_mean + daily_int_std,
    alpha=0.25, color=COLORS['main'],
    label='±1 Écart-type'
)
ax.fill_between(
    daily_int_mean.index,
    daily_int_mean - 2*daily_int_std,
    daily_int_mean + 2*daily_int_std,
    alpha=0.10, color=COLORS['purple'],
    label='±2 Écart-types'
)

ax.set_title('Global Intensity — Moyenne ± Écart-type journaliers (A)',
             fontweight='bold', fontsize=12)
ax.set_ylabel('Intensité (A)')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.show()

print(f'Intensité moyenne globale  : {daily_int_mean.mean():.4f} A')
print(f'Écart-type moyen           : {daily_int_std.mean():.4f} A')

In [ ]:
# === PATTERNS HORAIRES ET SAISONNIERS ===
df['hour']  = df.index.hour
df['month'] = df.index.month
df['year']  = df.index.year
df['dayofweek'] = df.index.dayofweek

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pattern horaire
hourly = df.groupby('hour')['Global_active_power'].mean()
axes[0].bar(hourly.index, hourly.values, color=COLORS['main'], edgecolor='white')
axes[0].set_title('Consommation moyenne par heure\nde la journée', fontweight='bold')
axes[0].set_xlabel('Heure'); axes[0].set_ylabel('kW moyen')
axes[0].set_xticks(range(0, 24, 2))

# Pattern mensuel
monthly = df.groupby('month')['Global_active_power'].mean()
month_names = ['Jan','Fév','Mar','Avr','Mai','Jun',
               'Jul','Aoû','Sep','Oct','Nov','Déc']
colors_m = [COLORS['warn'] if v > monthly.mean() else COLORS['second']
            for v in monthly.values]
axes[1].bar(monthly.index, monthly.values, color=colors_m, edgecolor='white')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names, rotation=30)
axes[1].axhline(monthly.mean(), color='gray', lw=1.5, linestyle='--', label='Moyenne')
axes[1].set_title('Consommation moyenne par mois\n(saisonnalité)', fontweight='bold')
axes[1].set_ylabel('kW moyen'); axes[1].legend()

# Pattern hebdomadaire
day_names = ['Lun','Mar','Mer','Jeu','Ven','Sam','Dim']
weekly_p = df.groupby('dayofweek')['Global_active_power'].mean()
colors_d = [COLORS['main'] if i < 5 else COLORS['amber'] for i in range(7)]
axes[2].bar(range(7), weekly_p.values, color=colors_d, edgecolor='white')
axes[2].set_xticks(range(7)); axes[2].set_xticklabels(day_names)
axes[2].set_title('Consommation moyenne par jour\nde la semaine', fontweight='bold')
axes[2].set_ylabel('kW moyen')

plt.suptitle('Patterns temporels de consommation électrique',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔄 Part 4 — Data Preprocessing for LSTM

In [ ]:
# === RÉÉCHANTILLONNAGE HORAIRE (pour réduire la taille) ===
# 2M lignes par minute → ~35K lignes par heure
df_hourly = df[['Global_active_power', 'Global_reactive_power',
                'Voltage', 'Global_intensity',
                'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].resample('H').mean()

# Supprimer les éventuels NaN créés par le resample
df_hourly.dropna(inplace=True)

print(f'Dataset rééchantillonné (horaire) : {df_hourly.shape[0]:,} lignes × {df_hourly.shape[1]} colonnes')
print(df_hourly.head())

In [ ]:
# === NORMALISATION (MinMaxScaler → [0, 1]) ===
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(df_hourly.values)

print(f'✅ Normalisation MinMaxScaler appliquée')
print(f'   Shape : {data_scaled.shape}')
print(f'   Min   : {data_scaled.min():.4f}')
print(f'   Max   : {data_scaled.max():.4f}')

# On prédit Global_active_power (colonne 0)
# On utilise toutes les colonnes comme features
n_features = data_scaled.shape[1]
print(f'\n   Nombre de features : {n_features}')
print(f'   Variable cible     : Global_active_power (colonne 0)')

In [ ]:
# === CRÉATION DES SÉQUENCES POUR LSTM ===
# L'LSTM prend en entrée des fenêtres temporelles de n_steps pas de temps
# Pour chaque séquence X de longueur n_steps, on prédit y (la valeur suivante)

def create_sequences(data, n_steps):
    """
    Crée des séquences d'entrée (X) et cibles (y) pour l'LSTM.
    X : (n_samples, n_steps, n_features)
    y : (n_samples,) — valeur de Global_active_power au pas suivant
    """
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i : i + n_steps, :])      # fenêtre de n_steps
        y.append(data[i + n_steps, 0])           # prochaine valeur GAP
    return np.array(X), np.array(y)

N_STEPS = 24   # 24 heures = 1 journée de contexte
X_seq, y_seq = create_sequences(data_scaled, N_STEPS)

print(f'✅ Séquences créées avec n_steps = {N_STEPS} heures')
print(f'   X shape : {X_seq.shape}  → (n_samples, n_steps, n_features)')
print(f'   y shape : {y_seq.shape}  → (n_samples,)')

In [ ]:
# === SPLIT TRAIN / TEST (80% / 20%) ===
TRAIN_RATIO = 0.8
split_idx = int(len(X_seq) * TRAIN_RATIO)

X_train = X_seq[:split_idx]
X_test  = X_seq[split_idx:]
y_train = y_seq[:split_idx]
y_test  = y_seq[split_idx:]

print('=== SPLITS FINAUX ===')
print(f'  X_train : {X_train.shape}  ({TRAIN_RATIO*100:.0f}% des données)')
print(f'  X_test  : {X_test.shape}   ({(1-TRAIN_RATIO)*100:.0f}% des données)')
print(f'  y_train : {y_train.shape}')
print(f'  y_test  : {y_test.shape}')
print(f'\n  Shape LSTM attendue : (batch_size, {N_STEPS}, {n_features})')
print(f'  ✅ Données prêtes pour l\'entraînement LSTM')

---
## 🧠 Part 5 — Building the LSTM Model

In [ ]:
# === ARCHITECTURE DU MODÈLE LSTM ===
model = keras.Sequential([
    # Couche LSTM 1 — return_sequences=True pour empiler une 2ème couche LSTM
    layers.LSTM(
        units=64,
        return_sequences=True,
        input_shape=(N_STEPS, n_features),
        name='lstm_1'
    ),
    layers.Dropout(0.2, name='dropout_1'),

    # Couche LSTM 2
    layers.LSTM(
        units=32,
        return_sequences=False,
        name='lstm_2'
    ),
    layers.Dropout(0.2, name='dropout_2'),

    # Couche Dense intermédiaire
    layers.Dense(16, activation='relu', name='dense_1'),

    # Couche de sortie — 1 neurone = prédiction de Global_active_power
    layers.Dense(1, activation='linear', name='output')
], name='LSTM_Power_Forecast')

# Compilation
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mae']
)

model.summary()

In [ ]:
# === VISUALISATION DE L'ARCHITECTURE ===
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 12); ax.set_ylim(0, 4); ax.axis('off')

blocks = [
    (0.5,  2.0, f'INPUT\n({N_STEPS}, {n_features})\nséquence horaire', '#B5D4F4'),
    (2.5,  2.0, 'LSTM 1\n64 units\nreturn_seq=True', '#1D9E75'),
    (4.2,  2.0, 'Dropout\n0.2', '#FAEEDA'),
    (5.8,  2.0, 'LSTM 2\n32 units\nreturn_seq=False', '#1D9E75'),
    (7.5,  2.0, 'Dropout\n0.2', '#FAEEDA'),
    (9.0,  2.0, 'Dense\n16 (ReLU)', '#534AB7'),
    (10.8, 2.0, 'OUTPUT\n1 neurone\n(GAP prédit)', '#E24B4A'),
]

for i, (x, y, label, color) in enumerate(blocks):
    rect = plt.Rectangle((x-0.7, y-0.7), 1.3, 1.4,
                           facecolor=color, edgecolor='white',
                           linewidth=2, zorder=3, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center',
            fontsize=8, fontweight='bold', zorder=4)
    if i < len(blocks)-1:
        ax.annotate('', xy=(blocks[i+1][0]-0.7, 2.0),
                    xytext=(x+0.6, 2.0),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=2))

ax.set_title(f'Architecture LSTM — Prédiction Global Active Power\n'
             f'Entrée : {N_STEPS}h × {n_features} features | Sortie : 1 valeur (prochaine heure)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Total paramètres : {model.count_params():,}')

---
## 🏋️ Part 6 — Training and Evaluating the LSTM Model

In [ ]:
# === ENTRAÎNEMENT DU MODÈLE LSTM ===
EPOCHS     = 20
BATCH_SIZE = 64

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=4,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=2, verbose=1, min_lr=1e-6
    )
]

print(f'🏋️  Entraînement LSTM — {EPOCHS} epochs max, batch={BATCH_SIZE}')
print(f'   Train : {X_train.shape[0]:,} séquences')
print(f'   GPU   : {len(tf.config.list_physical_devices("GPU")) > 0}\n')

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

print(f'\n✅ Entraînement terminé en {len(history.history["loss"])} epochs')

In [ ]:
# === COURBES DE LOSS — TRAIN vs VALIDATION ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history.history['loss']) + 1)

# MSE Loss
axes[0].plot(epochs_range, history.history['loss'],
             color=COLORS['main'], lw=2.5, marker='o', ms=5, label='Train Loss (MSE)')
axes[0].plot(epochs_range, history.history['val_loss'],
             color=COLORS['warn'], lw=2.5, marker='s', ms=5,
             linestyle='--', label='Val Loss (MSE)')
axes[0].fill_between(epochs_range,
                      history.history['loss'],
                      history.history['val_loss'],
                      alpha=0.1, color='gray')
best_ep = np.argmin(history.history['val_loss']) + 1
axes[0].axvline(best_ep, color='gray', lw=1.5, linestyle=':',
                label=f'Best epoch = {best_ep}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Validation Loss (MSE)', fontweight='bold')
axes[0].legend()

# MAE
axes[1].plot(epochs_range, history.history['mae'],
             color=COLORS['second'], lw=2.5, marker='o', ms=5, label='Train MAE')
axes[1].plot(epochs_range, history.history['val_mae'],
             color=COLORS['amber'], lw=2.5, marker='s', ms=5,
             linestyle='--', label='Val MAE')
axes[1].axvline(best_ep, color='gray', lw=1.5, linestyle=':',
                label=f'Best epoch = {best_ep}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE')
axes[1].set_title('Training & Validation MAE', fontweight='bold')
axes[1].legend()

plt.suptitle('Courbes d\'apprentissage — LSTM Power Forecast',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Meilleur epoch       : {best_ep}')
print(f'Meilleure val_loss   : {min(history.history["val_loss"]):.6f}')
print(f'Meilleure val_mae    : {min(history.history["val_mae"]):.6f}')

In [ ]:
# === ÉVALUATION SUR LE TEST SET ===
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

# Prédictions
y_pred_scaled = model.predict(X_test, verbose=0).flatten()

# Dénormalisation (reconstruction de la colonne GAP)
def inverse_gap(scaled_vals, scaler, n_features):
    """Inverse MinMaxScaler pour la colonne Global_active_power (index 0)"""
    dummy = np.zeros((len(scaled_vals), n_features))
    dummy[:, 0] = scaled_vals
    return scaler.inverse_transform(dummy)[:, 0]

y_true_real = inverse_gap(y_test, scaler, n_features)
y_pred_real = inverse_gap(y_pred_scaled, scaler, n_features)

# Métriques sur les valeurs réelles (kW)
rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
mae  = mean_absolute_error(y_true_real, y_pred_real)
mape = np.mean(np.abs((y_true_real - y_pred_real) / (y_true_real + 1e-8))) * 100

print('=' * 55)
print('ÉVALUATION — LSTM TEST SET')
print('=' * 55)
print(f'  Test MSE (normalisé) : {test_loss:.6f}')
print(f'  Test MAE (normalisé) : {test_mae:.6f}')
print(f'  RMSE (kW réels)      : {rmse:.4f} kW')
print(f'  MAE  (kW réels)      : {mae:.4f} kW')
print(f'  MAPE                 : {mape:.2f}%')
print(f'  Moyenne réelle       : {y_true_real.mean():.4f} kW')
print(f'  Erreur relative      : {mae/y_true_real.mean()*100:.2f}% de la moyenne')
print('=' * 55)

In [ ]:
# === VISUALISATION DES PRÉDICTIONS ===
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

# 1. Vue complète test set
axes[0].plot(y_true_real, color=COLORS['main'], lw=0.8,
             alpha=0.7, label='Réel')
axes[0].plot(y_pred_real, color=COLORS['warn'], lw=0.8,
             alpha=0.7, label='Prédit')
axes[0].set_title('Prédictions LSTM vs Réalité — Test Set complet', fontweight='bold')
axes[0].set_ylabel('Global Active Power (kW)')
axes[0].legend()

# 2. Zoom sur les 7 premiers jours (168 heures)
zoom = 168
axes[1].plot(range(zoom), y_true_real[:zoom],
             color=COLORS['main'], lw=2, marker='o', ms=2, label='Réel')
axes[1].plot(range(zoom), y_pred_real[:zoom],
             color=COLORS['warn'], lw=2, marker='s', ms=2,
             linestyle='--', label='Prédit')
axes[1].fill_between(range(zoom), y_true_real[:zoom], y_pred_real[:zoom],
                      alpha=0.2, color='gray', label='Erreur')
axes[1].set_title(f'Zoom — 7 premiers jours du test set (168 heures)', fontweight='bold')
axes[1].set_ylabel('kW'); axes[1].set_xlabel('Heure')
axes[1].legend()

# 3. Scatter réel vs prédit
axes[2].scatter(y_true_real[::10], y_pred_real[::10],
                alpha=0.3, color=COLORS['purple'],
                edgecolors='none', s=20)
min_v = min(y_true_real.min(), y_pred_real.min())
max_v = max(y_true_real.max(), y_pred_real.max())
axes[2].plot([min_v, max_v], [min_v, max_v], 'r--', lw=2, label='Prédiction parfaite')
axes[2].set_title(f'Réel vs Prédit — RMSE={rmse:.4f} kW  MAE={mae:.4f} kW  MAPE={mape:.1f}%',
                   fontweight='bold')
axes[2].set_xlabel('Valeur réelle (kW)')
axes[2].set_ylabel('Valeur prédite (kW)')
axes[2].legend()

plt.suptitle('LSTM — Prédiction de Global Active Power',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === ANALYSE DES RÉSIDUS ===
residuals = y_true_real - y_pred_real

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Distribution des résidus
axes[0].hist(residuals, bins=60, color=COLORS['main'], edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', lw=2, linestyle='--')
axes[0].axvline(residuals.mean(), color=COLORS['amber'], lw=2,
                linestyle='-.', label=f'Moyenne={residuals.mean():.4f}')
axes[0].set_title('Distribution des résidus', fontweight='bold')
axes[0].set_xlabel('Résidu (kW)'); axes[0].legend()

# Résidus dans le temps
axes[1].plot(residuals[::10], color=COLORS['warn'], lw=0.6, alpha=0.7)
axes[1].axhline(0, color='black', lw=1.5, linestyle='--')
axes[1].axhline(residuals.std(), color='gray', lw=1, linestyle=':', label='+1σ')
axes[1].axhline(-residuals.std(), color='gray', lw=1, linestyle=':', label='-1σ')
axes[1].set_title('Résidus dans le temps', fontweight='bold')
axes[1].set_xlabel('Index'); axes[1].set_ylabel('Résidu (kW)')
axes[1].legend()

# Erreur absolue par quantile de consommation
abs_err = np.abs(residuals)
quantiles = pd.qcut(y_true_real, q=5, labels=['Très faible','Faible','Moyenne','Élevée','Très élevée'])
err_by_q = pd.Series(abs_err).groupby(quantiles).mean()
axes[2].bar(err_by_q.index, err_by_q.values,
            color=COLORS['purple'], edgecolor='white')
axes[2].set_title('Erreur absolue moyenne\npar niveau de consommation', fontweight='bold')
axes[2].set_ylabel('MAE (kW)')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Analyse des résidus — LSTM Power Forecast',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Résidus — Moyenne : {residuals.mean():.6f} kW')
print(f'Résidus — Std     : {residuals.std():.4f} kW')
print(f'% résidus < 0.5 kW : {(abs_err < 0.5).mean()*100:.1f}%')
print(f'% résidus < 1.0 kW : {(abs_err < 1.0).mean()*100:.1f}%')

---
## 📋 Conclusion

### Résumé du pipeline

| Étape | Action |
|---|---|
| Import | Chargement du .txt (`;` séparateur, `?` = NaN) |
| Missing values | Remplacement par la **moyenne** de chaque colonne |
| Rééchantillonnage | Minute → Heure (réduction 60× de la taille) |
| Normalisation | MinMaxScaler [0, 1] sur toutes les features |
| Séquences | Fenêtres glissantes de **24 heures** |
| Split | 80% train / 20% test |
| Architecture | LSTM(64) → Dropout → LSTM(32) → Dropout → Dense(16) → Dense(1) |
| Entraînement | Adam, MSE loss, EarlyStopping, ReduceLROnPlateau |

### Performances
Le modèle apprend à prédire la consommation électrique de la prochaine heure à partir des 24 heures précédentes. L'LSTM capture les patterns journaliers (pics matin/soir) et les tendances saisonnières.

### Améliorations possibles
- **Fenêtre plus longue** : 48h ou 168h (1 semaine) pour capturer les patterns hebdomadaires
- **Prédiction multi-step** : prédire les 24 prochaines heures d'un coup
- **Attention mechanism** : Transformer ou LSTM avec attention pour de meilleures performances
- **Features externes** : température, heure du jour, jour férié

---
*Notebook réalisé dans le cadre du DI Bootcamp — UCI Household Power Consumption LSTM*